In [ ]:
# Loading data
import pandas as pd
import numpy as np
dfe = pd.read_csv("202425_all_state_funded_pupils_characteristics_and_geography_breakdowns_revised.csv")
imd = pd.read_csv('File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv')
pd.set_option('display.max_rows', None)

In [ ]:
# Initial data exploration (only DfE dataset)
print(dfe.shape)
#print(dfe.head())
print(dfe.columns.tolist())
print(dfe["geographic_level"].value_counts())
print(dfe["breakdown_topic"].value_counts())


In [ ]:
# columns to keep:
cols = [
    'time_period',
    'new_la_code',
    'la_name',
    'school_count',
    'pupil_count',
    'attainment8_average',
    'progress8_average',
    'progress8_pupil_percent',
    'ebacc_aps_average',
    'ebacc_entering_percent',
    'ebacceng_aps_average',
    'ebaccmat_aps_average',
    'ebaccsci_aps_average',
    'ebacchum_aps_average',
    'ebacclan_aps_average',
    'valueaddedsci_average',
    'valueaddedhum_average',
    'valueaddedlan_average',
    'breakdown_topic'
]
# The DfE dataset contains multiple geographic levels, National, Regional, Local Authority
# For this project, only Local Authority level records are retained:
dfe_loc=dfe[dfe["geographic_level"] == "Local authority"].copy() # keeping only data for 'Local authority' level
dfe_loc=dfe_loc[cols]

In [ ]:
# initial cleaning Dfe
dfe_loc.isna().sum().sort_values(ascending=False) # result - no Null values, instead we have 'z' values
z_counts = (dfe_loc == 'z').sum()
z_counts[z_counts > 0].sort_values(ascending=False) # where and how many z we have
dfe_loc = dfe_loc.replace('z', np.nan) #replace z with nulls to be able to change datatype to numeric
dfe_loc.dtypes # some columns, which are supposed to be numeric, are objects or strings
numeric_cols = [
    'school_count',
    'pupil_count',
    'attainment8_average',
    'progress8_average',
    'progress8_pupil_percent',
    'ebacc_aps_average',
    'ebacc_entering_percent',
    'ebacceng_aps_average',
    'ebaccmat_aps_average',
    'ebaccsci_aps_average',
    'ebacchum_aps_average',
    'ebacclan_aps_average',
    'valueaddedsci_average',
    'valueaddedhum_average',
    'valueaddedlan_average'
]

for col in numeric_cols:
    dfe_loc[col] = pd.to_numeric(dfe_loc[col])

# 'time period' has int64 dtype - changing  to str:
dfe_loc['time_period'] = dfe_loc['time_period'].astype(str)

dfe_loc.dtypes

In [ ]:
#Initial data exploration (IMD dataset)
imd.shape
imd.columns.to_list()

In [ ]:
# columns to keep: 
imd_cols = [
    'Local Authority District code (2019)',
    'Local Authority District name (2019)',
    'Index of Multiple Deprivation (IMD) Score',
    'Income Score (rate)',
    'Employment Score (rate)',
    'Education, Skills and Training Score',
    'Health Deprivation and Disability Score',
    'Crime Score',
    'Barriers to Housing and Services Score',
    'Living Environment Score',
    'Income Deprivation Affecting Children Index (IDACI) Score (rate)',
]
imd = imd[imd_cols]
imd.isna().sum() 
imd.describe()

The DfE dataset is reported at Local Authority level, while the IMD dataset is reported at Local Authority District level. The geographies must be aligned before the datasets can be merged.

In [ ]:
# what LA we have in DfE:
authorities_dfe = dfe_loc[["new_la_code", "la_name"]].drop_duplicates().sort_values("la_name").reset_index(drop=True)
authorities_dfe

In [ ]:
# what LA District we have in IMD:
authorities_imd = imd[['Local Authority District code (2019)', 'Local Authority District name (2019)']].drop_duplicates().sort_values("Local Authority District name (2019)").reset_index(drop=True)
authorities_imd

In [ ]:
dfe_codes = set(dfe_loc["new_la_code"].dropna().astype(str).str.strip())
imd_codes = set(imd["Local Authority District code (2019)"].dropna().astype(str).str.strip())

matching_codes = dfe_codes & imd_codes
only_in_dfe = dfe_codes - imd_codes
only_in_imd = imd_codes - dfe_codes

print("DFE codes:", len(dfe_codes))
print("IMD codes:", len(imd_codes))
print("Matching:", len(matching_codes))
print("Only in DFE:", len(only_in_dfe))
print("Only in IMD:", len(only_in_imd))

only_dfe_df = dfe_loc[dfe_loc["new_la_code"].astype(str).str.strip().isin(only_in_dfe)
][["new_la_code", "la_name"]].drop_duplicates().sort_values("la_name")

only_dfe_df.to_csv("only_in_dfe.csv", index=False)

only_imd_df = imd[imd["Local Authority District code (2019)"].astype(str).str.strip().isin(only_in_imd)
][["Local Authority District code (2019)", "Local Authority District name (2019)"]
].drop_duplicates().sort_values("Local Authority District name (2019)")

only_imd_df.to_csv("only_in_imd.csv", index=False)

matching_df = dfe[dfe["new_la_code"].astype(str).str.strip().isin(matching_codes)
][["new_la_code", "la_name"]]\
.drop_duplicates()\
.sort_values("la_name")

matching_df.to_csv("matching_codes.csv", index=False)



DFE codes: 157
IMD codes: 317
Matching: 124
Only in DFE: 33
Only in IMD: 193


In [ ]:
# Since there are different geo levels in datasets, we need look up table which match la codes with la/district codes, 
# downloaded from https://geoportal.statistics.gov.uk/datasets/fdddaa25a0df4679b496620557cca534_0/explore
df_lookup = pd.read_csv('Look_up_2019.csv')
df_lookup.columns.to_list()
df_lookup.head()

In [ ]:
#Merge IMD district data with lookup
imd_lookup = imd.merge(
    df_lookup,
    left_on="Local Authority District code (2019)",
    right_on="LAD19CD",
    how="left"
)

In [ ]:
#checking the merging result
imd_lookup.head()
imd_lookup["LAD19CD"].isna().sum()
imd["Local Authority District code (2019)"].nunique() 
imd_lookup["LAD19CD"].nunique() # checking that IMD and merged dataset have the same amount of LA districts

In [149]:
# aggregate imd to LA level
imd_la = (
    imd_lookup.groupby(["CTYUA19CD", "CTYUA19NM"], as_index=False).agg({
        "Index of Multiple Deprivation (IMD) Score": "mean",
        "Income Score (rate)": "mean",
        "Employment Score (rate)": "mean",
        "Education, Skills and Training Score": "mean",
        "Health Deprivation and Disability Score": "mean",
        "Crime Score": "mean",
        "Barriers to Housing and Services Score": "mean",
        "Living Environment Score": "mean",
        "Income Deprivation Affecting Children Index (IDACI) Score (rate)": "mean"
    })
)

In [ ]:

#The IMD dataset uses 2019 geography, while the DfE dataset uses more recent authority codes.
# Several authorities were reorganised after 2019 and require manual mapping.

authority_map = {
    "E06000060": "E10000002",  # Buckinghamshire <- old Buckinghamshire
    "E06000061": "E10000021",  # North Northamptonshire <- Northamptonshire
    "E06000062": "E10000021",  # West Northamptonshire <- Northamptonshire
    "E06000063": "E10000006",  # Cumberland <- Cumbria
    "E06000064": "E10000006",  # Westmorland and Furness <- Cumbria
    "E06000065": "E10000023",  # North Yorkshire <- old North Yorkshire
    "E06000066": "E10000027",  # Somerset <- old Somerset
}
dfe_loc["merge_la_code"] = dfe_loc["new_la_code"].replace(authority_map)

In [ ]:
# now we can merge DfE with IMD
final = dfe_loc.merge(
    imd_la,
    left_on="merge_la_code",
    right_on="CTYUA19CD",
    how="left"
)

In [ ]:

final[final["Index of Multiple Deprivation (IMD) Score"].isna()][["new_la_code", "la_name"]].drop_duplicates()

In [ ]:
# dataset for q1, keep only these columns:
cols_total = [
    'time_period',
    'new_la_code',
    'la_name',
    'school_count',
    'pupil_count',
    'attainment8_average',
    'progress8_average',
    'progress8_pupil_percent',
    'ebacc_aps_average',
    'ebacc_entering_percent',
]
dfe_total = dfe_loc[(dfe_loc["breakdown_topic"] == "Total")].copy()
dfe_total = dfe_total[cols_total].copy()
dfe_total.groupby(['time_period', 'la_name']).size().value_counts() # To check that after filtering to the total breakpoint we have only one row for each pair la+year



In [ ]:
# dataset for q3:
cols_subjects = [
    'time_period',
    'new_la_code',
    'la_name',
    'ebacceng_aps_average',
    'ebaccmat_aps_average',
    'ebaccsci_aps_average',
    'ebacchum_aps_average',
    'ebacclan_aps_average',
    'valueaddedsci_average',
    'valueaddedhum_average',
    'valueaddedlan_average'
]
dfe_subjects = dfe_loc[(dfe_loc["breakdown_topic"] == "Total")].copy()
dfe_subjects = dfe_subjects[cols_subjects]


In [ ]:
# q4&q5
dfe_gender = dfe_loc[dfe_loc["breakdown_topic"] == "Sex"].copy()
dfe_fsm = dfe_loc[dfe_loc["breakdown_topic"] == "FSM status"].copy()
dfe_disadv = dfe_loc[dfe_loc["breakdown_topic"] == "Disadvantage status"].copy()
dfe_lang = dfe_loc[dfe_loc["breakdown_topic"] == "First language"].copy()
#dfe_prior = dfe_loc[dfe_loc["breakdown_topic"] == "KS2 scaled score group"].copy() // do we need this??
#dfe_ethnicity = dfe_loc[dfe_loc["breakdown_topic"] == "Ethnicity"].copy() // could be difficult, a lot of splitting
# columns for these datasets??